# Benchmark — Optimizador Multiobjetivo de Layouts

Interfaz interactiva para ejecutar y analizar el benchmark del módulo `mo_module`.  
Configura los parámetros con los controles y pulsa **▶ Ejecutar benchmark**.

In [4]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path().resolve().parent))

In [5]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np

from src.mo_module.benchmark import (
    BenchmarkRunner,
    get_default_circuits,
    make_custom_circuit,
    analyze_results,
    DEFAULT_BENCHMARK_CIRCUITS,
)
from src.mo_module.optimizer import OptimizerConfig

print('✓ Imports OK')

✓ Imports OK


In [ ]:

# ──────────────────────────────────────────────
#  Panel de configuración
# ──────────────────────────────────────────────

CIRCUIT_DESCRIPTIONS = {
    bc.name: f"{bc.name}  —  {bc.description}"
    for bc in DEFAULT_BENCHMARK_CIRCUITS
}

# ipywidgets SelectMultiple: options como (label, value)
# → mostramos la descripción larga, guardamos el nombre corto
CIRCUIT_OPTIONS = [(desc, name) for name, desc in CIRCUIT_DESCRIPTIONS.items()]

# --- Selección de circuitos ---
w_circuits = widgets.SelectMultiple(
    options=CIRCUIT_OPTIONS,
    value=[name for _, name in CIRCUIT_OPTIONS],   # todos por defecto
    rows=4,
    description='Circuitos',
    style={'description_width': '80px'},
    layout=widgets.Layout(width='620px'),
)

# --- Número de semillas ---
w_seeds = widgets.IntSlider(
    value=10,
    min=1, max=30, step=1,
    description='Semillas',
    style={'description_width': '80px'},
    layout=widgets.Layout(width='400px'),
    continuous_update=False,
)
w_seeds_label = widgets.Label(value='(30 para análisis completo, 10 para exploración)')

# --- Backend ---
w_backend = widgets.Dropdown(
    options=['fake_torino', 'fake_sherbrooke', 'fake_brisbane'],
    value='fake_torino',
    description='Backend',
    style={'description_width': '80px'},
    layout=widgets.Layout(width='300px'),
)

# --- Algoritmo ---
w_algo = widgets.RadioButtons(
    options=['nsga2', 'moead'],
    value='nsga2',
    description='Algoritmo',
    style={'description_width': '80px'},
)

# --- Población ---
w_pop = widgets.IntSlider(
    value=30,
    min=6, max=100, step=2,
    description='Población',
    style={'description_width': '80px'},
    layout=widgets.Layout(width='400px'),
    continuous_update=False,
)

# --- Generaciones ---
w_gens = widgets.IntSlider(
    value=50,
    min=5, max=200, step=5,
    description='Generaciones',
    style={'description_width': '80px'},
    layout=widgets.Layout(width='400px'),
    continuous_update=False,
)

# --- Botón de ejecución ---
w_run = widgets.Button(
    description='▶  Ejecutar benchmark',
    button_style='primary',
    icon='play',
    layout=widgets.Layout(width='220px', height='36px'),
)

# --- Área de progreso y resultados ---
w_status  = widgets.Label(value='Estado: en espera')
w_progress = widgets.IntProgress(
    value=0, min=0, max=1,
    description='Progreso:',
    bar_style='info',
    style={'description_width': '80px'},
    layout=widgets.Layout(width='500px', visibility='hidden'),
)
w_output = widgets.Output(layout=widgets.Layout(border='1px solid #ccc', padding='8px'))

# ──────────────────────────────────────────────
#  Layout del panel
# ──────────────────────────────────────────────
col_left = widgets.VBox([
    widgets.Label('Circuitos (Ctrl+clic para selección múltiple):'),
    w_circuits,
    widgets.HBox([w_seeds, w_seeds_label]),
    w_backend,
])
col_right = widgets.VBox([
    w_algo,
    w_pop,
    w_gens,
])
panel = widgets.HBox(
    [col_left, widgets.Box(layout=widgets.Layout(width='40px')), col_right],
    layout=widgets.Layout(border='1px solid #ddd', padding='12px', margin='8px 0'),
)

display(
    panel,
    widgets.HBox([w_run, widgets.Box(layout=widgets.Layout(width='16px')), w_status]),
    w_progress,
    w_output,
)


TraitError: Invalid selection: value not found

In [ ]:
# ──────────────────────────────────────────────
#  Lógica del botón
# ──────────────────────────────────────────────

_last_results = None   # guardado entre ejecuciones para la sección de gráficos
_last_report  = None

def on_run_clicked(b):
    global _last_results, _last_report

    selected_names = list(w_circuits.value)
    if not selected_names:
        w_status.value = '⚠ Selecciona al menos un circuito'
        return

    circuits = [bc for bc in DEFAULT_BENCHMARK_CIRCUITS if bc.name in selected_names]
    seeds = list(range(w_seeds.value))
    n_total = len(circuits) * len(seeds)

    # Resetear UI
    w_run.disabled = True
    w_run.description = '⏳ Ejecutando...'
    w_progress.max   = n_total
    w_progress.value = 0
    w_progress.layout.visibility = 'visible'
    w_output.clear_output()

    config = OptimizerConfig(
        algorithm=w_algo.value,
        population_size=w_pop.value,
        n_generations=w_gens.value,
        objectives=['depth', 'cnot_count'],
        verbose=False,
    )

    # ── Ejecutar con actualización de progreso por ejecución ──
    from src.qiskit_interface.backend_info import get_backend
    from src.mo_module.benchmark.runner import BenchmarkRun, BenchmarkResultSet
    from src.mo_module.optimizer import optimize_layout
    import time

    backend = get_backend(w_backend.value)
    result_set = BenchmarkResultSet(
        backend_name=w_backend.value,
        config=config,
    )
    t0 = time.perf_counter()

    run_idx = 0
    for bc in circuits:
        circuit = bc.create()
        for seed in seeds:
            run_idx += 1
            w_status.value = f'Ejecutando {bc.name}  seed={seed}  ({run_idx}/{n_total})'
            run_cfg = OptimizerConfig(
                algorithm=config.algorithm,
                population_size=config.population_size,
                n_generations=config.n_generations,
                objectives=list(config.objectives),
                optimization_level=config.optimization_level,
                seed=seed,
                verbose=False,
            )
            try:
                opt = optimize_layout(circuit=circuit, backend=backend, config=run_cfg)
                result_set.runs.append(BenchmarkRun(circuit_name=bc.name, seed=seed, result=opt))
            except Exception as exc:
                result_set.runs.append(BenchmarkRun(circuit_name=bc.name, seed=seed, error=str(exc)))
            w_progress.value = run_idx

    result_set.total_elapsed_s = time.perf_counter() - t0
    _last_results = result_set
    _last_report  = analyze_results(result_set)

    # ── Mostrar resumen ──
    with w_output:
        print(_last_results.summary())

    w_status.value = (
        f'✓ Completado: {result_set.n_ok} ok / {result_set.n_failed} fallidas '
        f'en {result_set.total_elapsed_s:.1f} s'
    )
    w_run.disabled    = False
    w_run.description = '▶  Ejecutar benchmark'


w_run.on_click(on_run_clicked)

---
## Informe estadístico detallado

Ejecuta la celda siguiente tras completar el benchmark para ver el informe completo.

In [ ]:
if _last_report is None:
    print('⚠ Ejecuta primero el benchmark.')
else:
    print(_last_report.to_text())

---
## Tabla resumen (pandas)

Tabla plana con una fila por circuito, exportable a CSV.

In [ ]:
if _last_report is None:
    print('⚠ Ejecuta primero el benchmark.')
else:
    df = pd.DataFrame(_last_report.to_dict()['rows'])
    display(df)
    # Descomenta para exportar:
    # df.to_csv('benchmark_resultados.csv', index=False)

---
## Visualización de resultados

Gráficas con los mejores valores de cada objetivo por semilla.

In [ ]:
if _last_results is None:
    print('⚠ Ejecuta primero el benchmark.')
else:
    circuit_names = _last_results.circuit_names
    n_circuits    = len(circuit_names)
    obj_names     = _last_results.runs_for_circuit(circuit_names[0])[0].result.objective_names
    n_obj         = len(obj_names)

    fig, axes = plt.subplots(
        n_obj, n_circuits,
        figsize=(4 * n_circuits, 3.5 * n_obj),
        squeeze=False,
    )
    fig.suptitle('Distribución de mejores valores por semilla', fontsize=13, fontweight='bold')

    for ci, cname in enumerate(circuit_names):
        for oi, oname in enumerate(obj_names):
            ax = axes[oi][ci]
            values = _last_results.best_per_seed(cname, oi)
            ax.boxplot(values, vert=True, patch_artist=True,
                       boxprops=dict(facecolor='#aec6e8', color='#3a7abf'),
                       medianprops=dict(color='#c0392b', linewidth=2))
            ax.set_title(f'{cname}\n{oname}', fontsize=9)
            ax.set_xticks([])
            ax.set_ylabel(oname if ci == 0 else '')
            # Superponer puntos individuales
            jitter = np.random.default_rng(0).uniform(-0.1, 0.1, len(values))
            ax.scatter([1 + j for j in jitter], values, alpha=0.6, s=18, color='#2c3e50', zorder=3)

    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.show()

In [ ]:
# Evolución de la mediana del frente de Pareto por circuito
if _last_results is None:
    print('⚠ Ejecuta primero el benchmark.')
else:
    circuit_names = _last_results.circuit_names
    obj_names     = _last_results.runs_for_circuit(circuit_names[0])[0].result.objective_names

    fig, axes = plt.subplots(1, len(circuit_names), figsize=(5 * len(circuit_names), 4), squeeze=False)
    fig.suptitle('Frente de Pareto por semilla (mediana de cada semilla)', fontsize=12)

    for ci, cname in enumerate(circuit_names):
        ax = axes[0][ci]
        runs = _last_results.runs_for_circuit(cname)
        for run in runs:
            F = run.result.pareto_fitness
            if F is not None and len(F) > 0 and F.shape[1] >= 2:
                ax.scatter(F[:, 0], F[:, 1], alpha=0.4, s=14, color='#3a7abf')
        ax.set_xlabel(obj_names[0])
        ax.set_ylabel(obj_names[1] if len(obj_names) > 1 else '')
        ax.set_title(cname)
        ax.grid(True, alpha=0.3)

    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.show()

In [ ]:
# Comparativa de CV (coeficiente de variación) — estabilidad entre circuitos
if _last_report is None:
    print('⚠ Ejecuta primero el benchmark.')
else:
    cv_data = {}
    for ca in _last_report.circuit_analyses:
        for os in ca.objective_stats:
            cv_data.setdefault(os.name, {})[ca.circuit_name] = os.cv

    obj_names_r = list(cv_data.keys())
    if obj_names_r:
        fig, axes = plt.subplots(1, len(obj_names_r), figsize=(5 * len(obj_names_r), 4), squeeze=False)
        fig.suptitle('Coeficiente de variación (%) — menor = más estable', fontsize=12)

        for oi, oname in enumerate(obj_names_r):
            ax = axes[0][oi]
            cnames_ = list(cv_data[oname].keys())
            cvs_    = [cv_data[oname][c] for c in cnames_]
            colors  = ['#e74c3c' if v > 20 else '#2ecc71' for v in cvs_]
            bars    = ax.bar(cnames_, cvs_, color=colors, edgecolor='#2c3e50', alpha=0.85)
            ax.axhline(20, color='#e74c3c', linestyle='--', alpha=0.6, label='Umbral 20%')
            ax.set_ylabel('CV (%)')
            ax.set_title(oname)
            ax.legend(fontsize=8)
            for bar, v in zip(bars, cvs_):
                ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
                        f'{v:.1f}%', ha='center', va='bottom', fontsize=8)
            plt.setp(ax.get_xticklabels(), rotation=15, ha='right', fontsize=8)

        plt.tight_layout(rect=[0, 0, 1, 0.95])
        plt.show()